In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier
import random

In [2]:
# count = 0
# for i in range(len(data[72])):
#     if(data[72].values[i]==5):
#         print(i, '---->', data[72].values[i])
#         count += 1
#         if count == 5:
#             break
# # np.count_nonzero(data[72].values==9)

In [3]:
def load_data(mode='train'):
    path = "../../../../data/20191008/david_20191008_session"
    f_ext = ".mat"
    if mode=='train':
        mode_path = "1_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            file_data = loadmat(load_path)
            mdata = file_data['data']
            temp = pd.DataFrame(mdata)
            data = pd.concat([data, temp])
        return data
    elif mode=='test':
        mode_path = "2_"
        data = pd.DataFrame()
        for i in range(1, 11):
            value = str(i).zfill(2)
            load_path = path + mode_path + value + f_ext
            file_data = loadmat(load_path)
            mdata = file_data['data']
            temp = pd.DataFrame(mdata)
            data = pd.concat([data, temp])
        return data

In [4]:
def band_pass_filter(eeg, freq_range):
#     13 or 16
    info = mne.create_info(64, 512, ch_types=["eeg"] * 64)
    raw = mne.io.RawArray(eeg.T, info)
    
    iir_params = dict(order=5, ftype='cheby2', rs=2.)
    raw = raw.filter(freq_range[0], freq_range[1], fir_design='firwin', method='iir', iir_params=iir_params)

    return raw._data.T

In [5]:
def drop_classes(df):
    label_not_inc = list(range(2,9))
    indexes_to_drop = []
    i = 0
    while i < len(df):
        if df[72].values[i] in label_not_inc:
            list2 = list(range(i, 2048+i))
            i += 2048
            indexes_to_drop.extend(list2)
        else:
            i += 1
    indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
    df_sliced = df.take(list(indexes_to_keep))

    df_sliced = df_sliced.reset_index(drop=True)
    return df_sliced

In [6]:
def data_win(sfreq, data, asynch_label):
    sampling_window = 2 * sfreq
    shift_length = 1 * sfreq
    t_start = 0

    new_data = []
    labels = []

    while t_start + sampling_window < data.shape[0]:
        new_data.append(data[t_start:t_start+sampling_window, :].T)
        labels.append(asynch_label[t_start:t_start+sampling_window])

        t_start = t_start + shift_length

    return np.array(new_data), np.array(labels)

def transform_label(label_new):
    label = []
    for i in label_new:
        count1 = np.count_nonzero(i==1)
        count9 = np.count_nonzero(i==2)
        if count1 >= 512:
            to_add = 1
        elif count9 >= 512:
            to_add = 2
        else:
            to_add = 0
        label.append(to_add)
        
    label = np.array(label)
    return label

In [7]:
def prune_records(data_new, label):
    train_data = []
    label_data = []

    for i in range(len(label)):
        if label[i] != 0:
            label_data.append(label[i])
            train_data.append(data_new[i])
    label_data = np.array(label_data)
    train_data = np.array(train_data)

    return train_data, label_data

In [8]:
def fbcsp(df, labels, sfreq, train=True, csp_objects=None):
    freq = 4
    increment = 4
    end_freq = 40
    if train==True:
        csp_objects = []
        csp_data = []
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)
            final_data = csp.fit_transform(X, y)

            csp_objects.append(csp)
            csp_data.append(final_data)

        return np.array(csp_objects), np.array(csp_data), y

    else:
        csp_data = []
        count = 0
        while freq < end_freq:
            freq_range = []
            freq_range.append(freq)
            freq_range.append(freq+increment)
            freq += increment
            out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

            X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
            y = transform_label(y)
            X, y = prune_records(X, y)

            final_data = csp_objects[count].transform(X)
            count += 1

            csp_data.append(final_data)

        return np.array(csp_data), y

In [9]:
def convert_class(sfreq, trigger_points, df):
    asynch_label = []
    i = 0
    while i < len(df[72]):
        if(df[72].values[i] == 1):
            end = (sfreq * trigger_points[1]) + 1
            for j in range(i, i+end):
                asynch_label.append(1)
            i += end
        elif(df[72].values[i] == 9):
            end = (sfreq * trigger_points[9]) + 1
            for j in range(i, i+end):
                asynch_label.append(2)
            i += end
        elif(df[72].values[i] == 0):
            asynch_label.append(0)
            i += 1

    return np.array(asynch_label)

In [10]:
def itr(n_class, p_class, c_time):
    B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

    return B

def performance_metrics(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    print('Accuracy Score: ', acc)
    print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
    print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
    print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [11]:
def augment_data(df, asynch_label, n_times=5, split_size=2):
    random.seed(42)
    copy_df = df
    copy_df = copy_df.drop([72], axis = 1)
    copy_df[72] = asynch_label

    unique = np.unique(asynch_label)
    flag = 0

    for val in range(len(unique)):
        if unique[val] != 0:
            print('Generating new data for label', unique[val])
            temp_df = copy_df[copy_df[72] == unique[val]]
            temp_df.reset_index(drop=True)

            divided = []
            i = 0
            
            while i < len(temp_df):
                temp = temp_df.iloc[i:i+2048, :].values
                divided.append(temp)
                i += 2048

            divided = np.array(divided)

            k = split_size
            num_of_times = n_times
            size = len(divided[0]) // k

            for i in range(num_of_times):
                parts = []
                end = len(temp_df) // 2048
#                 print(divided.shape)
                choices_main = random.sample(range(0, end), 2)
                parts.append(divided[choices_main[0]][:size, :])
                parts.append(divided[choices_main[0]][size:, :])
                parts.append(divided[choices_main[1]][:size, :])
                parts.append(divided[choices_main[1]][size:, :])

                choices_final = random.sample(range(0, len(parts)), k)
                new_data_1 = np.concatenate((parts[choices_final[0]], parts[choices_final[1]]))
                new_data_2 = np.concatenate((parts[(len(parts) + choices_final[0]) % 4], parts[(len(parts) + choices_final[1]) % 4]))

                if flag == 0:
                    augmented_data = np.concatenate((new_data_1, new_data_2))
                    flag = 1
                elif flag == 1:
                    augmented_data = np.concatenate((augmented_data, new_data_1, new_data_2))

            # print(augmented_data.shape)

    return augmented_data

In [12]:
def get_train_data():
    sfreq = 512
    trigger_points = {1:4, 9:4}
    
    df = load_data(mode='train')
    train_df = drop_classes(df)
    
    train_df = train_df.drop([64, 65, 66, 67, 68, 69, 70, 71], axis=1)
    asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
    
    print(train_df)

    augmented_data = augment_data(train_df, asynch_label, n_times=250, split_size=2)
    extra_label = augmented_data[:, -1]

    augmented_data = pd.DataFrame(augmented_data)
    augmented_data = augmented_data.rename(columns = {64: 72})
    train_df = pd.concat([train_df, augmented_data])

    print(train_df)

    asynch_label = np.concatenate((asynch_label, extra_label))
    
    
    csp_objects, csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=True)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y, csp_objects

def get_eval_data(csp_objects):
    sfreq = 512
    trigger_points = {1:4, 9:4}
    
    df = load_data(mode='test')
    train_df = drop_classes(df)
    train_df = train_df.drop([64, 65, 66, 67, 68, 69, 70, 71], axis=1)
    
    asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
    csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=False, csp_objects=csp_objects)
    
    final_data = pd.DataFrame(csp_data[0])
    col_count = 4
    for i in range(1, len(csp_data)):
        for j in range(len(csp_data[i].T)):
            final_data[str(col_count)] = csp_data[i].T[j]
            col_count += 1

    return final_data.values, y

In [13]:
def train_mibif(X, y):
    print('------Train---------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
    select_K = SelectKBest(mutual_info_classif,k=10).fit(X, y)
    extra = select_K.get_support()
    if extra[0] == True:
        extra[1] = True
    for i in range(2, len(extra)):
        if extra[i] == True:
            if i%2 == 0:
                extra[i+1] = True
            else:
                extra[i-1] = True
    pos = np.where(extra==False)
    New_train = np.delete(X_train, list(pos[0]), 1)
    New_test = np.delete(X_test, list(pos[0]), 1)
    # New_train=select_K.transform(X_train)
    # New_test=select_K.transform(X_test)
    ss = StandardScaler()
    New_train = ss.fit_transform(New_train,y_train)
    New_test = ss.transform(New_test)

    print('####### SVM#####')
    svm = SVC()
    svm.fit(New_train, y_train)
    y_pred = svm.predict(New_test)
    performance_metrics(y_test, y_pred)

    print('##########LDA#########')
    lda = LinearDiscriminantAnalysis()
    lda.fit(New_train, y_train)
    y_pred = lda.predict(New_test)
    performance_metrics(y_test, y_pred)

    return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
    print('------Test--------')
    X = np.delete(X, pos, 1)
    X = ss.transform(X)
    print('#####SVM######')
    y_pred = svm.predict(X)
    performance_metrics(y, y_pred)
    print('#####LDA######')
    y_pred = lda.predict(X)
    performance_metrics(y, y_pred)

In [14]:
def train_rf(X, y):
    print('--------Train--------')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
    clf = RandomForestClassifier(random_state=42)
    ss = StandardScaler()
    X_train = ss.fit_transform(X_train, y_train)
    clf.fit(X_train, y_train)
    X_test = ss.transform(X_test)
    y_pred = clf.predict(X_test)
    ####Random Forest#######
    performance_metrics(y_test, y_pred)

    return clf, ss
def eval_rf(clf, ss, X, y):
    print('--------Test------')
    X = ss.transform(X)
    y_pred = clf.predict(X)
    performance_metrics(y, y_pred)

In [15]:
X, y, csp_objects = get_train_data()
X_eval, y_eval = get_eval_data(csp_objects=csp_objects)

                0          1         2          3          4          5   \
0        -2.169347   3.305810  4.597246  25.940488  10.043918  15.377349   
1        -3.010009   2.161993  3.758405  25.886134   9.466555  14.792408   
2        -3.960744   0.989630  2.845109  25.790651   8.857406  14.211274   
3        -5.093204  -0.239647  1.785391  25.579338   8.157875  13.604070   
4        -6.305984  -1.415632  0.654858  25.301190   7.425998  13.066189   
...            ...        ...       ...        ...        ...        ...   
1288699 -17.026220 -25.993147  8.916349   5.662616  27.947708  75.133194   
1288700 -11.802690 -17.927489  8.741881   5.617850  25.057069  67.031284   
1288701  -6.561208 -10.745466  7.827329   5.146410  21.177437  56.469835   
1288702  -2.439279  -5.330363  6.358613   4.293887  16.608971  44.202850   
1288703  -0.027414  -1.972268  4.633519   3.215934  11.858147  31.508740   

                6          7          8          9   ...        55        56  \
0      

    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=64, n_times=3336704
    Range : 0 ... 3336703 =      0.000 ...  6516.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Computing data rank from raw with rank=None
    Using tolerance 97 (2.2e-16 eps * 64 dim * 6.8e+15  max singular value)
    Estimated rank (mag): 64
    MAG: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating covariance using EMPIRICAL
Done.
Computing data rank from raw with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 

- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=64, n_times=1121280
    Range : 0 ... 1121279 =      0.000 ...  2189.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=64, n_times=1121280
    Range : 0 ... 1121279 =      0.000 ...  2189.998 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 16 - 20 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 16.00, 20.00 Hz: -4.00, -4.00 dB

Cr

In [16]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.975
Cohen Kappa Score:  0.95
ITR (bits per minute):  24.94017205509989
Confusion Matrix:  [[525  15]
 [ 12 528]]
##########LDA#########
Accuracy Score:  0.9574074074074074
Cohen Kappa Score:  0.9148148148148149
ITR (bits per minute):  22.37833809637132
Confusion Matrix:  [[507  33]
 [ 13 527]]
------Test--------
#####SVM######
Accuracy Score:  0.6680497925311203
Cohen Kappa Score:  0.33768464445207824
ITR (bits per minute):  2.492805178757873
Confusion Matrix:  [[115   5]
 [ 75  46]]
#####LDA######
Accuracy Score:  0.8132780082987552
Cohen Kappa Score:  0.6268708068123172
ITR (bits per minute):  9.163094035269141
Confusion Matrix:  [[110  10]
 [ 35  86]]


In [17]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.9861111111111112
Cohen Kappa Score:  0.9722222222222222
ITR (bits per minute):  26.832268908744147
Confusion Matrix:  [[531   9]
 [  6 534]]
--------Test------
Accuracy Score:  0.7302904564315352
Cohen Kappa Score:  0.46159133871799274
ITR (bits per minute):  4.768573779534407
Confusion Matrix:  [[115   5]
 [ 60  61]]
